In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [14]:
train = pd.read_csv('datasets/mortgage_train.csv')
test = pd.read_csv('datasets/mortgage_test.csv')
display(train.head())
display(test.head())
print("Shape of x_train: ", train.shape)
print("Shape of x_test: ", test.shape)

,default,DTI,LTV,Score,UnempShock,PriorDelinq
0,0,38.4,77.7,595,0,1
1,0,36.5,79.2,703,0,0
2,1,43.5,85.1,718,0,0
3,1,39.4,91.4,748,0,0
4,1,44.9,73.7,693,0,1


,default,DTI,LTV,Score,UnempShock,PriorDelinq
0,0,36.8,87.2,797,0,0
1,0,26.9,85.5,667,0,0
2,0,34.4,80.2,743,0,0
3,0,27.7,82.5,653,0,0
4,0,30.8,92.0,641,0,1


Shape of x_train:  (250, 6)
Shape of x_test:  (100, 6)


In [15]:
X_train = sm.add_constant(train.drop('default', axis=1))
y_train = train['default']
X_test = sm.add_constant(test.drop('default', axis=1))
y_test = test['default']


### i) Logistic regression + gradient descent

In [22]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, lr=0.01, iterations=5000):
    weights = np.zeros(X.shape[1])
    for _ in range(iterations):
        z = np.dot(X, weights)
        predictions = sigmoid(z)
        gradient = np.dot(X.T, (predictions - y)) / len(y)
        weights -= lr * gradient
    return weights

In [23]:
betas = gradient_descent(X_train.values, y_train.values)
print("Estimated coefficients: ", betas)

Estimated coefficients:  [-0.046778    7.51515402  8.769135   -1.21430636  0.34460168  0.72398   ]


In [31]:
logit = sm.Logit(y_train, X_train).fit()
print("Logit coefficients: \n",logit.params)

Optimization terminated successfully.
         Current function value: 0.313874
         Iterations 7
Logit coefficients: 
 const         -9.638606
DTI            0.039820
LTV            0.063833
Score          0.001081
UnempShock     0.756488
PriorDelinq    1.106445
dtype: float64


### ii) Thresholds 0.5 and 0.30

In [33]:
from sklearn.metrics import *

In [ ]:
test_probs = logit.predict(X_test)

def evaluate_performance(probs, actual, threshold):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(actual, preds)
    acc = accuracy_score(actual, preds)
    prec = precision_score(actual, preds, zero_division=0)
    rec = recall_score(actual, preds, zero_division=0)
    
    print(f"\n--- Metrics for Threshold: {threshold} ---")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Accuracy:  {acc}")
    print(f"Precision: {prec}")
    print(f"Recall:    {rec:.4f}")

evaluate_performance(test_probs, y_test, 0.50)
evaluate_performance(test_probs, y_test, 0.30)


--- Metrics for Threshold: 0.5 ---
Confusion Matrix:
[[88  0]
 [11  1]]
Accuracy:  0.89
Precision: 1.0
Recall:    0.0833

--- Metrics for Threshold: 0.3 ---
Confusion Matrix:
[[85  3]
 [10  2]]
Accuracy:  0.87
Precision: 0.4
Recall:    0.1667


### iv) OLS

In [40]:
ols_model = sm.OLS(y_train, X_train).fit()
ols_preds = ols_model.predict(X_test)
print("OLS coefficients: \n", ols_model.params)

OLS coefficients: 
 const         -0.537774
DTI            0.003728
LTV            0.005574
Score          0.000091
UnempShock     0.072590
PriorDelinq    0.152119
dtype: float64
